For best results, run this notebook with 2x T4 GPUs

In [ ]:
import os
import base64
import subprocess
from kaggle_secrets import UserSecretsClient

In [ ]:
# set this to match your Kaggle notebook's URL slug (also used as the Drive folder name)
NOTEBOOK_NAME = "your-kaggle-notebook"
assert NOTEBOOK_NAME != "your-kaggle-notebook", (
    "Set NOTEBOOK_NAME to your real notebook slug - the placeholder pollutes /kaggle/working and Drive."
)

In [ ]:
repo = f"/kaggle/working/{NOTEBOOK_NAME}"

if not os.path.exists(repo):
    subprocess.run(
        ["git", "clone", "https://github.com/noshou/APS360.git", repo],
        check=True,
    )
else:
    # Force-sync to origin/main instead of `git pull` (merge): /kaggle/working/{repo} is a
    # scratch checkout regenerated each session, never hand-edited, so any local diff (e.g.
    # leftover files from a path that moved upstream) should just be discarded rather than
    # block the pull with a merge conflict.
    subprocess.run(["git", "-C", repo, "fetch", "origin", "main"], check=True)
    subprocess.run(
        ["git", "-C", repo, "reset", "--hard", "origin/main"], check=True
    )

In [ ]:
# only needs to be run once per session
%pip install -q xraydb beartype jaxtyping hdf5plugin h5py pyyaml torch-tb-profiler
!curl -fsSL https://rclone.org/install.sh | sudo bash

# per-epoch diagnostic plots (Train/eval_plots.py -> Baselines/metrics.py) render text
# through real LaTeX (xelatex) with JuliaMono as the font -- classic latex+dvipng
# (matplotlib's usetex default) can't load an arbitrary system font, only
# xelatex/lualatex + fontspec can. Same toolchain Baselines/kaggle_baselines.ipynb
# installs. Skip this if RunConfig.data_dir is left unset.
!sudo apt-get update -q && sudo apt-get install -y -q texlive-xetex texlive-latex-recommended texlive-fonts-recommended
!mkdir -p ~/.fonts && curl -fsSL https://github.com/cormullion/juliamono/releases/latest/download/JuliaMono-ttf.tar.gz \
    | tar -xz -C ~/.fonts && fc-cache -f ~/.fonts

In [ ]:
# ── rclone / Google Drive setup ── run once per session ─────────────────────
# Mirrors checkpoints off-box so a session timeout doesn't lose them.
# Prereq (one-time): configure rclone locally, then paste the contents of
# base64 -w0 ~/.config/rclone/rclone.conf into a Kaggle Secret named RCLONE_CONF
# (Add-ons -> Secrets). REMOTE_NAME is derived automatically from your rclone config.
# we use base64 so we can copy/paste in single line even tho it's multiline token :)
# Self-contained imports (don't rely on the earlier imports cell having run in
# this pass - Kaggle's "Run All" has occasionally executed cells out of order).

import os, base64, subprocess
from kaggle_secrets import UserSecretsClient

conf_path = "/kaggle/working/rclone.conf"
with open(conf_path, "w") as f:
    f.write(
        base64.b64decode(
            UserSecretsClient().get_secret("RCLONE_CONF")
        ).decode()
    )
os.environ["RCLONE_CONFIG"] = conf_path

remotes = (
    subprocess.run(["rclone", "listremotes"], capture_output=True, text=True)
    .stdout.strip()
    .split("\n")
)
remote = remotes[0] if remotes and remotes[0] else ""
# Guard: an empty remote makes REMOTE_NAME a *local* relative path, so rclone
# would silently copy checkpoints into /kaggle/working instead of Drive.
assert remote.endswith(":"), (
    f"No rclone remote found (rclone listremotes -> {remotes!r}). "
    "Check the RCLONE_CONF secret. Without a remote, checkpoints would be written "
    "LOCALLY to /kaggle/working, not Drive."
)
REMOTE_NAME = remote + f"{NOTEBOOK_NAME}/ckpts/"

out = subprocess.run(
    ["rclone", "mkdir", REMOTE_NAME], capture_output=True, text=True
)
print(
    "remote drive ──>",
    REMOTE_NAME,
    "(ok)" if out.returncode == 0 else f"ERROR: {out.stderr.strip()}",
)

# Everything the run records goes to a sibling data/ subfolder: one dir per epoch
# (metrics.json, per-batch loss curve, loss-per-epoch curve, diagnostic plots) plus
# run_config.rtf at its root.
DATA_REMOTE = remote + f"{NOTEBOOK_NAME}/data/"
out = subprocess.run(
    ["rclone", "mkdir", DATA_REMOTE], capture_output=True, text=True
)
print(
    "remote data  ──>",
    DATA_REMOTE,
    "(ok)" if out.returncode == 0 else f"ERROR: {out.stderr.strip()}",
)

In [ ]:
# ── Verbosity ─────────────────────────────────────────────────────────────────
#
#   "epoch"      - one summary line per epoch (train/val/test loss + R²)
#   "batch"      - also prints every 20 batches, in training AND in the end-of-epoch
#                  val/test/plots passes ("[val]", "[test]", "[test/plots]"). Those
#                  passes are as many batches as a training epoch, so at "epoch"
#                  verbosity the run looks hung for a long time after the last batch.
#   "diagnostic" - per-batch NaN/Inf check with full tensor stats on the first
#                  10 batches; use when debugging numerical issues
#
VERBOSITY = "batch"

import sys, os

sys.path.insert(0, f"/kaggle/working/{NOTEBOOK_NAME}")
os.environ["PYTORCH_ALLOC_CONF"] = (
    "expandable_segments:True"  # prevents fragmentation
)
os.environ["PYTHONPATH"] = (
    f"/kaggle/working/{NOTEBOOK_NAME}"  # inherited by mp.spawn workers
)

# clear python cache
for mod in list(sys.modules.keys()):
    if "ScatterNet" in mod or "train" in mod:
        del sys.modules[mod]

from ScatterNet.utils.config import RunConfig, DEFAULT_BUCKETS
from Train.train import main

In [ ]:
# ── Fresh training run ──────────────────────────────────────────────────────
# Full training (profiler OFF). Tune chunk sizes below.
# To PROFILE instead, use the commented diagnostic cell further down (sets
# profiler=True).

cfg = RunConfig(
    # --- paths ---
    hdf5="/kaggle/input/datasets/noso0s0n/iql50/I(q)L50.h5",
    encodings_sqlite3_path="/kaggle/input/datasets/noso0s0n/iql50/iq_train_set-ENCODING.sqlite3",
    ckpt_best="/kaggle/working/scatternet_best.pt",
    ckpt_dir="/kaggle/working/checkpoints",
    resume=None,
    data_dir="/kaggle/working/scatternet_data",  # everything the run records,
    # one dir per epoch ({data_dir}/epoch_NNN/): metrics.json
    # (that epoch's train/val/test loss + R2), loss_per_batch.png,
    # loss_per_epoch.png, and baseline-style diagnostic plots
    # (per-q R2, per-q percent error, Kratky overlay, residual
    # histogram, error-vs-atom-count) -- see Train/eval_plots.py
    # + Baselines/metrics.py. Nothing is written per-run except
    # run_config.rtf at the root, so a resume can't clobber an
    # earlier epoch's numbers; concatenate the per-epoch
    # metrics.json files afterwards for the full history.
    # Comment out (or set None) to skip -- needs the LaTeX/
    # JuliaMono toolchain installed above.
    # --- checkpointing / crash-safety ---
    ckpt_rclone_dest=f"{REMOTE_NAME}",
    data_rclone_dest=f"{DATA_REMOTE}",  # off-box copy of data_dir (Drive)
    ckpt_interval_sec=600,
    # --- model ---
    lambda_1=128,
    lambda_2=4,  # 4 message-passing rounds, each with its OWN
    # proj_agg/sigbilin/rms_norm weights (not shared across
    # rounds anymore -- see README MessagePass section).
    lambda_3=256,  # OutputHead hidden width (widened from 64 -- narrow
    # tapering was flagged as a capacity bottleneck). NOTE:
    # OutputHead's per-chunk bilinear-output tensor
    # (N, atm_chunk, Q, lambda_3) is NOT checkpointed like
    # MessagePass's chem_env is, so this directly multiplies
    # peak activation memory on the heaviest atom-count
    # buckets -- watch peak_alloc/reserved on the first few
    # batches after restart, and drop atm_chunk if it OOMs.
    lambda_4=4,
    lambda_5=128,
    msg_seed=42,
    atm_chunk=512,  # 1024/512 OOM'd on 2xT4 (12.13/14.56G used +3.19G
    mol_chunk=256,  # alloc). 800/400 also tried (post RMSNorm-autocast +
    # .contiguous() recompile fixes): same ~880ms/batch steady-
    # state as 512/256 but ~55% more peak mem (8.9G vs 5.8G) and
    # bigger compile stalls - compute is chunk-invariant here, so
    # bigger chunks bought nothing. 512/256 is the chosen point,
    # see README Appendix A1. Chosen before the lambda_3=256 widen,
    # re-check peak memory now that OutputHead's activation is 4x.
    dp_atom_threshold=101,  # calibrated for 2x T4 on this dataset (1.8x faster, peak
    # still 13.55G). Routes high-N/low-M buckets to molecule-split
    # DP (skips TP's mid-forward all-reduce). Two guards define the
    # safe band: strict M < threshold keeps the M=101 bucket on TP
    # (memory upper bound), and N >= 2*mol_chunk keeps small-N
    # buckets on TP (no DP benefit). 0 = always TP (old behaviour).
    # Don't push into the thousands (OOM risk) -- see README §7.
    compile=True,  # torch.compile Embed/MessagePass/OutputHead's checkpointed step fns
    amp=True,  # fp16 autocast + GradScaler (T4 fp16 tensor cores); RFF projection stays fp32
    eps_embd=1e-8,
    eps_msgp=1e-3,
    # --- loss --- (RunConfig coerces these to float, so an int literal like `1`
    # won't crash later inside the beartype-guarded Loss.loss() call -- still write 1.0)
    lambda_6=1.0,
    lambda_7=0.15,  # sigma L2 penalty, weighted by ~q^2 across the grid
    # (normalized to mean 1, so this value means what it
    # did under the old flat penalty). Keeps the kernel
    # range long where low q needs it -- see README §8.
    # Lowered from 0.5: checkpoints showed _emb._sigma's
    # bilinear weights already suppressed toward triviality
    # (norm ~0.0024 vs ~3.2-3.4 for the structurally similar
    # _f0f1/_f2 weights), consistent with this penalty
    # overpowering sigma's intended per-atom/per-q expressiveness.
    # --- training ---
    lr=1.3e-4,
    lr_gamma=0.9,  # per-epoch ExponentialLR decay: lr *= 0.9 each epoch
    plateau_window=500,  # batches per rolling window for the reactive mid-epoch lr cut
    plateau_patience=3,  # non-improving windows before cutting lr by plateau_factor
    plateau_factor=0.8,  # multiplicative lr cut on a detected plateau; 1.0 = disabled
    weight_decay=0.1,  # AdamW decoupled decay, applied only to weight matrices
    # (biases/rms_norm/prelu/biasterm excluded) -- see Train/train.py
    # optimizer construction. Non-decoupled Adam+L2 with this same
    # value previously drove several params to numerical zero by
    # epoch 1 (dead message-passing gate + output-head layers); fixed
    # by switching to decoupled_weight_decay=True + the no-decay group.
    grad_clip=1.0,
    epochs=20,
    batcher_seed=0,
    atom_size_ceil=30230,
    dataset_frac=1.0,  # fraction of EACH split's batches to use, (0.0, 1.0].
    # Lower (e.g. 0.1) for a quick end-to-end smoke run.
    # Applies to train, val AND test: val/test hold one batch
    # per bucket just like train, so epoch-end eval costs about
    # a train epoch -- thinning train alone just lets eval
    # dominate. Deterministic off batcher_seed, fixed for the
    # whole run. 1.0 = full dataset (default, no-op).
    num_workers=3,
    max_batches=None,
    verbosity=VERBOSITY,
    # --- data ---
    buckets=DEFAULT_BUCKETS,
)

main(cfg)

In [ ]:
# # ── PROFILER / diagnostic run ───────────────────────────────────────────────
# # Uncomment to profile instead of train. Stops after the profiling window
# # (no eval/checkpoint). Every rank is wrapped in torch.profiler (per-rank
# # TensorBoard trace) AND a lightweight section timer that prints, per rank, a
# # data-wait / H2D / forward / backward / grad-allreduce / clip / step breakdown
# # plus the heaviest batches - compare ranks to spot tensor-parallel skew.

# PROF_FIXTURES = [
#     ('rcsb_med', '1EKP'),      # 6046 atoms  [heavy_m]
#     ('rcsb_med', '3WNM'),      # 6046 atoms  [heavy_m]
#     ('rcsb_med', '2VDX'),      # 6046 atoms  [heavy_m]
#     ('rcsb_med', '1I80'),      # 6046 atoms  [heavy_m]
#     ('rcsb_med', '3R0P'),      # 6046 atoms  [heavy_m]
#     ('rcsb_med', '4XDO'),      # 6046 atoms  [heavy_m]
#     ('rcsb_med', '1FSW'),      # 6046 atoms  [heavy_m]
#     ('rcsb_med', '4KA8'),      # 6045 atoms  [heavy_m]
#     ('rcsb_med', '2ZEF'),      # 6045 atoms  [heavy_m]
#     ('rcsb_med', '5A7Q'),      # 6044 atoms  [heavy_m]
#     ('rcsb_med', '3U6B'),      # 6044 atoms  [heavy_m]
#     ('rcsb_med', '3PZB'),      # 6043 atoms  [heavy_m]
#     ('rcsb_med', '4LAF'),      # 6043 atoms  [heavy_m]
#     ('rcsb_med', '4D6D'),      # 6043 atoms  [heavy_m]
#     ('rcsb_med', '1IJQ'),      # 6043 atoms  [heavy_m]
#     ('rcsb_med', '4WSI'),      # 6042 atoms  [heavy_m]
#     ('rcsb_med', '4Z7G'),      # 6042 atoms  [heavy_m]
#     ('rcsb_med', '1QD1'),      # 6042 atoms  [heavy_m]
#     ('rcsb_med', '4K8K'),      # 6042 atoms  [heavy_m]
#     ('rcsb_med', '1LGC'),      # 6042 atoms  [heavy_m]
#     ('rcsb_med', '4UZ6'),      # 6042 atoms  [heavy_m]
#     ('rcsb_med', '4L9L'),      # 6041 atoms  [heavy_m]
#     ('rcsb_med', '2YOF'),      # 6041 atoms  [heavy_m]
#     ('rcsb_med', '1UOO'),      # 6040 atoms  [heavy_m]
#     ('rcsb_med', '3ZYZ'),      # 6040 atoms  [heavy_m]
#     ('rcsb_med', '2COG'),      # 6040 atoms  [heavy_m]
#     ('rcsb_med', '3S88'),      # 6040 atoms  [heavy_m]
#     ('rcsb_med', '1YEO'),      # 4698 atoms  [heavy_nm]
#     ('rcsb_med', '1B9J'),      # 4698 atoms  [heavy_nm]
#     ('rcsb_med', '2XUW'),      # 4698 atoms  [heavy_nm]
#     ('rcsb_med', '2E4T'),      # 4698 atoms  [heavy_nm]
#     ('rcsb_med', '2OGW'),      # 4698 atoms  [heavy_nm]
#     ('rcsb_med', '4FL5'),      # 4698 atoms  [heavy_nm]
#     ('rcsb_med', '1VNG'),      # 4698 atoms  [heavy_nm]
#     ('rcsb_med', '3BNG'),      # 4698 atoms  [heavy_nm]
#     ('rcsb_med', '1EAW'),      # 4698 atoms  [heavy_nm]
#     ('rcsb_med', '4I3S'),      # 4698 atoms  [heavy_nm]
#     ('rcsb_med', '4KA0'),      # 4698 atoms  [heavy_nm]
#     ('rcsb_med', '3IWT'),      # 4698 atoms  [heavy_nm]
#     ('rcsb_med', '1DG3'),      # 4698 atoms  [heavy_nm]
#     ('rcsb_med', '2H7Y'),      # 4698 atoms  [heavy_nm]
#     ('rcsb_med', '1ITC'),      # 4698 atoms  [heavy_nm]
#     ('rcsb_med', '5BXD'),      # 4698 atoms  [heavy_nm]
#     ('rcsb_med', '3G5M'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '1GD2'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '1GZT'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '3IO5'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '1MXG'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '4DCJ'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '2X4H'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '4UZU'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '1SQN'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '1SXH'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '3FW3'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '3H16'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '4G9C'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '1GQF'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '3I5Y'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '4IHL'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '4EHA'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '4RCA'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '2F37'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '3OM2'),      # 4287 atoms  [heavy_nm]
#     ('rcsb_med', '4WZY'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '13YT'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '13YS'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '13XK'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '13ZY'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '3NHS'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '4XDH'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '4EGD'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '3OWH'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '4YI0'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '4HFR'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '3ZP0'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '1JSH'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '3C3I'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '13ZM'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '2GJJ'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '13ZR'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '13ZE'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '13ZF'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '3KTY'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '4BUU'),      # 4053 atoms  [heavy_nm]
#     ('rcsb_med', '4G92'),      # 3760 atoms  [heavy_nm]
#     ('rcsb_med', '3TQY'),      # 3760 atoms  [heavy_nm]
#     ('rcsb_med', '1IJK'),      # 3760 atoms  [heavy_nm]
#     ('COD', 'COD_80241'),      # 3760 atoms  [heavy_nm]
#     ('rcsb_med', '4GWO'),      # 3760 atoms  [heavy_nm]
#     ('rcsb_med', '1RC6'),      # 3760 atoms  [heavy_nm]
#     ('rcsb_med', '2QA2'),      # 3760 atoms  [heavy_nm]
#     ('COD', 'COD_205788'),     # 3760 atoms  [heavy_nm]
#     ('COD', 'COD_49879'),      # 3760 atoms  [heavy_nm]
#     ('rcsb_med', '2Y3W'),      # 3760 atoms  [heavy_nm]
#     ('rcsb_med', '3BU8'),      # 3760 atoms  [heavy_nm]
#     ('rcsb_med', '1RUL'),      # 3760 atoms  [heavy_nm]
#     ('rcsb_med', '4M81'),      # 3760 atoms  [heavy_nm]
#     ('rcsb_med', '2BUZ'),      # 3760 atoms  [heavy_nm]
#     ('COD', 'COD_399960'),     # 3760 atoms  [heavy_nm]
#     ('rcsb_med', '2AAW'),      # 3760 atoms  [heavy_nm]
#     ('COD', 'COD_491880'),     # 3760 atoms  [heavy_nm]
#     ('COD', 'COD_58938'),      # 3760 atoms  [heavy_nm]
#     ('rcsb_med', '4XXC'),      # 3760 atoms  [heavy_nm]
#     ('COD', 'COD_299627'),     # 3296 atoms  [heavy_nm]
#     ('rcsb_med', '1MAM'),      # 3296 atoms  [heavy_nm]
#     ('COD', 'COD_364764'),     # 3296 atoms  [heavy_nm]
#     ('rcsb_med', '3KBM'),      # 3296 atoms  [heavy_nm]
#     ('rcsb_med', '1FO9'),      # 3296 atoms  [heavy_nm]
#     ('rcsb_med', '4E1C'),      # 3296 atoms  [heavy_nm]
#     ('COD', 'COD_76793'),      # 3296 atoms  [heavy_nm]
#     ('rcsb_med', '4G50'),      # 3296 atoms  [heavy_nm]
#     ('rcsb_med', '3M6P'),      # 3296 atoms  [heavy_nm]
#     ('rcsb_med', '3E6H'),      # 3296 atoms  [heavy_nm]
#     ('rcsb_med', '1Y7E'),      # 3296 atoms  [heavy_nm]
#     ('rcsb_med', '4GZP'),      # 3296 atoms  [heavy_nm]
#     ('rcsb_med', '2WU6'),      # 3296 atoms  [heavy_nm]
#     ('rcsb_med', '4YLA'),      # 3296 atoms  [heavy_nm]
#     ('rcsb_med', '2C7X'),      # 3296 atoms  [heavy_nm]
#     ('COD', 'COD_38738'),      # 3296 atoms  [heavy_nm]
#     ('mofs', 'core_RAHPAT'),   # 3296 atoms  [heavy_nm]
#     ('rcsb_med', '3CGG'),      # 3296 atoms  [heavy_nm]
#     ('rcsb_med', '4GSY'),      # 3296 atoms  [heavy_nm]
#     ('rcsb_med', '2YU2'),      # 3296 atoms  [heavy_nm]
#     ('rcsb_med', '2ASM'),      # 3296 atoms  [heavy_nm]
#     ('rcsb_med', '2XE8'),      # 3296 atoms  [heavy_nm]
#     ('rcsb_med', '1A5F'),      # 3264 atoms  [heavy_nm]
#     ('COD', 'COD_215144'),     # 3264 atoms  [heavy_nm]
#     ('rcsb_med', '3UCS'),      # 3264 atoms  [heavy_nm]
#     ('rcsb_med', '3T2Y'),      # 3264 atoms  [heavy_nm]
#     ('rcsb_med', '2H62'),      # 3264 atoms  [heavy_nm]
#     ('rcsb_med', '2DUL'),      # 3264 atoms  [heavy_nm]
#     ('rcsb_med', '3O63'),      # 3264 atoms  [heavy_nm]
#     ('rcsb_med', '4B8E'),      # 3264 atoms  [heavy_nm]
#     ('COD', 'COD_397983'),     # 3264 atoms  [heavy_nm]
#     ('COD', 'COD_371052'),     # 3264 atoms  [heavy_nm]
#     ('rcsb_med', '3ME9'),      # 3264 atoms  [heavy_nm]
#     ('rcsb_med', '1Q66'),      # 3264 atoms  [heavy_nm]
#     ('rcsb_med', '3T9E'),      # 3264 atoms  [heavy_nm]
#     ('rcsb_med', '4YXR'),      # 3264 atoms  [heavy_nm]
#     ('rcsb_med', '4JID'),      # 3264 atoms  [heavy_nm]
#     ('COD', 'COD_217259'),     # 3264 atoms  [heavy_nm]
#     ('mofs', 'core_ULEGOH'),   # 3264 atoms  [heavy_nm]
#     ('COD', 'COD_19212'),      # 3264 atoms  [heavy_nm]
#     ('rcsb_med', '4JE8'),      # 3264 atoms  [heavy_nm]
#     ('COD', 'COD_371043'),     # 3264 atoms  [heavy_nm]
#     ('rcsb_med', '3K7N'),      # 3264 atoms  [heavy_nm]
#     ('COD', 'COD_292663'),     # 3264 atoms  [heavy_nm]
#     ('COD', 'COD_315967'),     # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_46601'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_377476'),     # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73478'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73485'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73483'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73474'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73491'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_190834'),     # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73481'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73475'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_322179'),     # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_367165'),     # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_270734'),     # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_78153'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_305070'),     # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73502'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_191294'),     # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73470'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73500'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_473503'),     # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_357391'),     # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73492'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_53733'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73476'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_225713'),     # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73498'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_396379'),     # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73472'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_396380'),     # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_225737'),     # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73486'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73496'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_349418'),     # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_249202'),     # 2160 atoms  [heavy_nm]
#     ('mofs', 'core_XILLEL'),   # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_340643'),     # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73487'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_201236'),     # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73490'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73499'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_73489'),      # 2160 atoms  [heavy_nm]
#     ('COD', 'COD_116401'),     # 684 atoms  [regular]
#     ('COD', 'COD_105910'),     # 684 atoms  [regular]
#     ('COD', 'COD_114020'),     # 684 atoms  [regular]
#     ('COD', 'COD_122987'),     # 684 atoms  [regular]
#     ('COD', 'COD_133446'),     # 684 atoms  [regular]
#     ('COD', 'COD_127831'),     # 684 atoms  [regular]
#     ('COD', 'COD_144858'),     # 684 atoms  [regular]
#     ('COD', 'COD_12384'),      # 684 atoms  [regular]
#     ('COD', 'COD_165754'),     # 684 atoms  [regular]
#     ('COD', 'COD_160799'),     # 684 atoms  [regular]
#     ('COD', 'COD_164868'),     # 684 atoms  [regular]
#     ('COD', 'COD_165996'),     # 684 atoms  [regular]
#     ('COD', 'COD_171429'),     # 684 atoms  [regular]
#     ('COD', 'COD_175131'),     # 684 atoms  [regular]
#     ('COD', 'COD_169476'),     # 684 atoms  [regular]
#     ('COD', 'COD_168062'),     # 684 atoms  [regular]
#     ('COD', 'COD_185628'),     # 684 atoms  [regular]
#     ('COD', 'COD_189689'),     # 684 atoms  [regular]
#     ('COD', 'COD_18891'),      # 684 atoms  [regular]
#     ('COD', 'COD_181067'),     # 684 atoms  [regular]
#     ('COD', 'COD_199389'),     # 684 atoms  [regular]
#     ('COD', 'COD_199507'),     # 684 atoms  [regular]
#     ('COD', 'COD_196958'),     # 684 atoms  [regular]
#     ('COD', 'COD_192323'),     # 684 atoms  [regular]
#     ('mofs', 'core_WOBQEL'),   # 304 atoms  [regular]
#     ('mofs', 'core_RAZXIA02'), # 304 atoms  [regular]
#     ('mofs', 'core_YARBIE'),   # 304 atoms  [regular]
#     ('mofs', 'core_XUMSOP'),   # 304 atoms  [regular]
#     ('mofs', 'core_WARFAY03'), # 304 atoms  [regular]
#     ('mofs', 'core_SABVUN'),   # 304 atoms  [regular]
#     ('mofs', 'core_VEJYUF'),   # 304 atoms  [regular]
#     ('mofs', 'core_TUYJAZ'),   # 304 atoms  [regular]
#     ('mofs', 'core_ULAQAB'),   # 304 atoms  [regular]
# ]  # 222 molecules  |  heavy_nm=7  heavy_m=7  regular=7

# cfg = RunConfig(

#     # --- paths ---
#     hdf5                    = "/kaggle/input/datasets/noso0s0n/iql50/I(q)L50.h5",
#     encodings_sqlite3_path  = "/kaggle/input/datasets/noso0s0n/iql50/iq_train_set-ENCODING.sqlite3",
#     ckpt_best               = "/kaggle/working/scatternet_best.pt",
#     ckpt_dir                = "/kaggle/working/checkpoints",
#     resume                  = None,

#     # --- checkpointing / crash-safety ---
#     ckpt_rclone_dest     = f"{REMOTE_NAME}",
#     ckpt_interval_sec    = 600,

#     # --- model ---
#     lambda_1             = 128,
#     lambda_2             = 4,
#     lambda_3             = 256,
#     lambda_4             = 4,
#     lambda_5             = 128,
#     msg_seed             = 42,
#     atm_chunk            = 512,  # 512/256 chosen - see README Appendix A1
#     mol_chunk            = 256,
#     dp_atom_threshold    = 101,     # calibrated value (see "Fresh training run" cell + README
#                                     # §7). Set 0 to profile the always-TP baseline for comparison.
#     compile              = True,    # torch.compile Embed/MessagePass/OutputHead's checkpointed step fns
#     amp                  = True,    # fp16 autocast + GradScaler (T4 fp16 tensor cores); RFF projection stays fp32
#     eps_embd             = 1e-8,
#     eps_msgp             = 1e-3,

#     # --- loss ---
#     lambda_6             = 1.0,
#     lambda_7             = 0.15,

#     # --- training ---
#     lr                   = 1.3e-4,
#     weight_decay         = 0.1,
#     grad_clip            = 1.0,
#     epochs               = 20,
#     batcher_seed         = 0,
#     atom_size_ceil       = 30230,
#     num_workers          = 3,
#     max_batches          = None,
#     verbosity            = VERBOSITY,

#     # --- data ---
#     buckets              = DEFAULT_BUCKETS,

#     # --- profiler (diagnostic run) ---
#     profiler             = True,  # per-rank torch.profiler + section timers
#     prof_warmup          = 2,     # warmup batches (profiled, discarded)
#     prof_active          = 1000,  # recorded batches; bump for representative stats
#     prof_molecules       = PROF_FIXTURES,  # hardcoded fixture list (stable across DB changes)
# )

# os.environ["TORCH_LOGS"] = "recompiles"  # print a reason each time Dynamo recompiles
# os.environ["CUDA_MODULE_LOADING"] = "EAGER"  # skip lazy module loading - free, no downside
# main(cfg)

In [ ]:
# # ═══════════════════════════════════════════════════════════════════════════
# # RESUME after a crash / session timeout
# # ═══════════════════════════════════════════════════════════════════════════
# # Training saves a numbered resume checkpoint to Drive every ckpt_interval_sec
# # (checkpoint_<epoch>_<batch>.pt, plus checkpoint_<epoch>_final.pt at each epoch
# # boundary) -- every save is its own file, none overwrite each other, so the
# # whole history survives a crash for later comparison. To resume:
# #   1. Re-run the cells above: deps -> git pull -> rclone setup -> the
# #      "Fresh training run" cell, but with its trailing `main(cfg)` line
# #      commented out (that cell is what defines `cfg`; left as-is it would
# #      start a fresh run from scratch).
# #   2. Run THIS cell. It finds the newest checkpoint on Drive, pulls just that
# #      one file, and continues from its saved batch. Mid-epoch resume is exact
# #      (the per-epoch seed makes the shuffle reproducible), so you lose at most
# #      ckpt_interval_sec of work.
# import dataclasses, re, subprocess

# # REMOTE_NAME already ends in "/". ckpt_dir is a directory of many
# # checkpoint_<epoch>_<batch>.pt files (batch is either an int or the literal
# # "final" for an epoch-boundary save), so list it rather than assuming one
# # fixed filename.
# ckpt_dir_remote = REMOTE_NAME.rstrip("/") + "/"

# ls = subprocess.run(["rclone", "lsf", ckpt_dir_remote], capture_output=True, text=True)
# names = [n.strip() for n in ls.stdout.splitlines() if n.strip()]
# ckpt_re = re.compile(r"^checkpoint_(\d+)_(\d+|final)\.pt$")
# candidates = [(n, ckpt_re.match(n)) for n in names]
# candidates = [(n, m) for n, m in candidates if m]

# # A missing/empty listing fails clearly rather than silently starting fresh.
# assert candidates, (
#     f"No checkpoints at {ckpt_dir_remote} (rclone lsf -> {ls.stdout!r} stderr "
#     f"{ls.stderr.strip()!r}). The previous run never pushed one: the first "
#     f"checkpoint is written ckpt_interval_sec seconds into training. Check what "
#     f"is there with: rclone lsf {ckpt_dir_remote}"
# )

# # Pick the checkpoint with the highest (epoch, batch), treating "final" as
# # sorting AFTER any numeric batch within the same epoch (an epoch-boundary
# # save for epoch E is later than any mid-epoch save of epoch E, but earlier
# # than any checkpoint of epoch E+1).
# def _sort_key(item):
#     _, m = item
#     epoch = int(m.group(1))
#     batch = m.group(2)
#     batch_key = (1, 0) if batch == "final" else (0, int(batch))
#     return (epoch, batch_key)

# chosen_filename, _ = max(candidates, key=_sort_key)

# # pull just the chosen checkpoint back from Drive
# subprocess.run(
#     ["rclone", "copy", ckpt_dir_remote + chosen_filename, "/kaggle/working/"],
#     check=True,
# )

# resume_cfg = dataclasses.replace(cfg, resume=f"/kaggle/working/{chosen_filename}")
# main(resume_cfg)
